In [8]:
import pdal
import geopandas as gpd
import sys
# sys.path.append('../../src/streaming/3dep/get_als')
import pandas as pd
from torch.utils.data import DataLoader


In [2]:
from dataset import ALSCookieDataset, load_als_cookie, identity

many of the bbs tiles are coming up with no returned point cloude even in the 'matched' data product

In [3]:

bbs = pd.read_parquet('../../datasets/BBS/BBS_train_test.parquet')
butterflies = pd.read_csv('../../datasets/butterflies/species_observations.csv')
bfsites = butterflies.groupby(['event.id'])[['product_name_AWS','lat','lon']].first()
bfsites.rename(columns={'lat':'Latitude','lon':'Longitude'}, inplace=True)

/tmp/ipykernel_40232/2299602145.py:2: DtypeWarning: Columns (0: date) have mixed types. Specify dtype option on import or set low_memory=False.
  butterflies = pd.read_csv('../../datasets/butterflies/species_observations.csv')


# test dataset and parallel dataloader speed

In [4]:
ds = ALSCookieDataset(site_table=bfsites, radius=100) 

In [5]:
# test dataset and dataloader speeds

2.5s per 100m radius cookie


parallel dataloader workers do produce speed-up (tested with 4 workers here)
- throughput ~1.3 patches/sec
- batch size of 4: 3.7 s/iter w 4 workers -> ~11 patches/sec with 40 workers


implies 20 workers -> 6 patches/s throughput or perhaps 40 workers -> 12 patches/second

> cannot use fork for DataLoader (or need to avoid conflict over tempdir) -> use 'spaw' multiprocessing_context


In [ ]:
%%timeit
ds[0]

In [9]:
dl = DataLoader(ds, batch_size=4, num_workers=4, collate_fn=identity, multiprocessing_context='spawn')

errors:

```
  what():  Could not read from 's3-us-west-2.amazonaws.com/usgs-lidar-public/USGS_LPC_CA_LosAngeles_2016_LAS_2018/ept-hierarchy/8-208-135-127.json'.
```

In [11]:
# %%timeit
from tqdm.autonotebook import tqdm
for i, batch in tqdm(enumerate(dl),total=32):
    if i>32:
        break

  0%|          | 0/32 [00:00<?, ?it/s]

# visualize point cloud

In [12]:
# select a random sample
# sample = bbs.sample(1).iloc[0] # Latitude, Longitude, product_name_AWS
sample = bfsites.sample(1).iloc[0] # lat, lon, product_name_AWS

In [13]:
pipeline = load_als_cookie(
    als_tile_name=sample['product_name_AWS'],
    lat=sample['Latitude'],
    lon=sample['Longitude'],
    point_crs='EPSG:4326',
    radius=100,
    out_crs=None,
)
len(pipeline.arrays)>0

True

In [14]:
points =pipeline.get_dataframe(0)
print(points.shape)
# relative coordinates
points['X'] = points['X'] - points['X'].min()
points['Y'] = points['Y'] - points['Y'].min()
points['Z'] = points['Z'] - points['Z'].min()
# clip intensity outliers
points['Return']=points.apply(
    lambda row: 'last' if (row['NumberOfReturns'] != 1 and row['ReturnNumber'] == row['NumberOfReturns']) else str(row['ReturnNumber']), axis=1
)

# points['Intensity_norm'] = points['Intensity'].clip(points['Intensity'].quantile(0.1), points['Intensity'].quantile(0.9))

(79034, 8)


In [15]:
import json
import pdal
import plotly.express as px
points["Intensity"] = points["Intensity"].astype(float)

fig = px.scatter_3d(
    points,
    x="X",
    y="Y",
    z="Z",
    color="Z", 
    # color_continuous_scale="Viridis",
    opacity=0.8,
)
fig.update_traces(marker={"size": 1})


# Enforce equal aspect ratio
fig.update_layout(scene=dict(aspectmode="data"))

# # print google maps link:
print(f"https://www.google.com/maps/search/?api=1&query={sample['Latitude']:.6f},{sample['Longitude']:.6f}")
fig.show()

https://www.google.com/maps/search/?api=1&query=42.798109,-85.984272
